# Context windows, tokens and what your prompt really costs

**The Agent Build Log · Build #005 · GenAI & LLM Foundations 5/8**

[Build #001](../01-tokens-and-embeddings/) showed that you pay per token. This build finds out **where the tokens
in a real bill come from**, measured on Claude Haiku 4.5:

1. [How data is formatted changes what it costs](#1-how-data-is-formatted-changes-what-it-costs)
2. [Chat history: the cost that compounds](#2-chat-history-the-cost-that-compounds)
3. [Output tokens are the expensive ones](#3-output-tokens-are-the-expensive-ones)
4. [What fits in the context window](#4-what-fits-in-the-context-window)
5. [Findings](#5-findings)

**Needs:** an Anthropic API key in a `.env` file. Most measurements use `messages.count_tokens` (no charge per call);
the rest cost a few cents on Claude Haiku 4.5.

In [1]:
import csv, io, json, os, statistics, time
from pathlib import Path

import pandas as pd
import yaml
from dotenv import load_dotenv
import anthropic

load_dotenv(Path.cwd() / ".env") or load_dotenv(Path.cwd().parents[1] / ".env")
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in a .env file (see README)"
pd.set_option("display.width", 150); pd.set_option("display.max_colwidth", 70)

MODEL = "claude-haiku-4-5"
PRICE = {"input": 1.00, "output": 5.00}    # $ per 1M tokens, Claude Haiku 4.5
CONTEXT = 200_000                          # Claude Haiku 4.5 context window (tokens)
client = anthropic.Anthropic()

def count(messages, system=None) -> int:
    kw = {"system": system} if system else {}
    return client.messages.count_tokens(model=MODEL, messages=messages, **kw).input_tokens

def cost(usage) -> float:
    return (usage.input_tokens * PRICE["input"] + usage.output_tokens * PRICE["output"]) / 1e6

TICKETS = [{"id": t["id"], "text": t["text"], **t["label"]} for t in json.loads(Path("data/tickets.json").read_text())["tickets"]]
print(f"{len(TICKETS)} tickets (reused from build #004) | model: {MODEL}")

20 tickets (reused from build #004) | model: claude-haiku-4-5


## 1. How data is formatted changes what it costs

The same 20 tickets, five ways. Same information, same model, different token counts. We count each one as a
user message and subtract the cost of an empty-ish message, so only the data is compared.

In [2]:
def as_csv(rows):
    buf = io.StringIO(); w = csv.DictWriter(buf, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows); return buf.getvalue()

def as_markdown(rows):
    cols = list(rows[0])
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join("" if r[c] is None else str(r[c]) for c in cols) + " |" for r in rows]
    return "\n".join(lines)

FORMATS = {
    "JSON, pretty (indent=2)": json.dumps(TICKETS, indent=2, ensure_ascii=False),
    "JSON, minified": json.dumps(TICKETS, separators=(",", ":"), ensure_ascii=False),
    "YAML": yaml.safe_dump(TICKETS, allow_unicode=True, sort_keys=False),
    "CSV": as_csv(TICKETS),
    "Markdown table": as_markdown(TICKETS),
}
base = count([{"role": "user", "content": "x"}])
fmt = pd.DataFrame([{"format": name, "characters": len(text),
                     "tokens": count([{"role": "user", "content": text}]) - base} for name, text in FORMATS.items()])
fmt["vs cheapest"] = (fmt["tokens"] / fmt["tokens"].min()).round(2)
fmt = fmt.sort_values("tokens").reset_index(drop=True)
fmt

,format,characters,tokens,vs cheapest
0,CSV,2749,863,1.00
1,Markdown table,3127,1046,1.21
2,"JSON, minified",4827,1393,1.61
3,YAML,4810,1584,1.84
4,"JSON, pretty (indent=2)",5908,2091,2.42


## 2. Chat history: the cost that compounds

An LLM API is stateless: **every turn re-sends the whole conversation**. We run a real 10-turn chat and log the
input tokens the API billed on each turn. Then we count (without calling the model) what each turn would have
sent if we had kept only the system prompt and the last 2 exchanges.

In [3]:
SYSTEM = ("You are a senior backend engineer helping a teammate design a URL shortener. "
          "Give practical, specific answers in a few short paragraphs.")
QUESTIONS = [
    "What are the core components of a URL shortener?",
    "How should I generate the short codes?",
    "Which database would you use and why?",
    "How do I handle 10,000 redirects per second?",
    "How do I stop people creating links to malware sites?",
    "How should custom aliases work?",
    "How do I track click analytics without slowing redirects?",
    "What happens when a link expires?",
    "How do I test all of this?",
    "Summarise the whole design in five bullet points.",
]

history, turns = [], []
for i, q in enumerate(QUESTIONS, 1):
    history.append({"role": "user", "content": q})
    t0 = time.perf_counter()
    m = client.messages.create(model=MODEL, max_tokens=400, system=SYSTEM, messages=history)
    answer = "".join(b.text for b in m.content if b.type == "text")
    history.append({"role": "assistant", "content": answer})
    turns.append({"turn": i, "input tokens (billed)": m.usage.input_tokens, "output tokens": m.usage.output_tokens,
                  "counted (full history)": count(history[:-1], system=SYSTEM),   # sanity check vs billed
                  "input if last 2 turns only": count(history[:-1][-5:], system=SYSTEM),  # last 2 exchanges + this question
                  "cost $": round(cost(m.usage), 6), "seconds": round(time.perf_counter() - t0, 2)})

chat = pd.DataFrame(turns)
totals = {"input tokens, full history": int(chat["input tokens (billed)"].sum()),
          "input tokens, last 2 turns only": int(chat["input if last 2 turns only"].sum()),
          "turn 1 input": int(chat["input tokens (billed)"].iloc[0]), "turn 10 input": int(chat["input tokens (billed)"].iloc[-1])}
totals["saving %"] = round((1 - totals["input tokens, last 2 turns only"] / totals["input tokens, full history"]) * 100)
print("count_tokens matches billed input on every turn:", bool((chat["counted (full history)"] == chat["input tokens (billed)"]).all()))
print(totals)
chat

count_tokens matches billed input on every turn: True
{'input tokens, full history': 18441, 'input tokens, last 2 turns only': 7261, 'turn 1 input': 46, 'turn 10 input': 3660, 'saving %': 61}


,turn,input tokens (billed),output tokens,counted (full history),input if last 2 turns only,cost $,seconds
0,1,46,381,46,46,0.001951,5.60
1,2,438,391,438,438,0.002393,5.47
2,3,840,377,840,840,0.002725,5.59
3,4,1234,400,1234,839,0.003234,5.52
4,5,1652,380,1652,855,0.003552,5.78
5,6,2041,390,2041,856,0.003991,5.69
6,7,2446,385,2446,841,0.004371,5.86
7,8,2841,391,2841,841,0.004796,5.50
8,9,3243,400,3243,844,0.005243,6.05
9,10,3660,247,3660,861,0.004895,4.03


## 3. Output tokens are the expensive ones

On Claude Haiku 4.5 an output token costs **5× an input token**. Same 5 questions, asked twice: once plainly, once
with a one-line instruction to be concise.

In [4]:
ASK = ["What is a database index?", "What is a race condition?", "What does idempotent mean for an API?",
       "What is the difference between a process and a thread?", "Why do we hash passwords?"]
CONCISE = "Answer in at most two sentences."

def run(question, system=None):
    kw = {"system": system} if system else {}
    t0 = time.perf_counter()
    m = client.messages.create(model=MODEL, max_tokens=800, messages=[{"role": "user", "content": question}], **kw)
    return {"input tokens": m.usage.input_tokens, "output tokens": m.usage.output_tokens,
            "cost $": cost(m.usage), "seconds": time.perf_counter() - t0}

rows = []
for q in ASK:
    rows.append({"question": q, "style": "no instruction", **run(q)})
    rows.append({"question": q, "style": "concise instruction", **run(q, CONCISE)})
out_df = pd.DataFrame(rows)
verbosity = out_df.groupby("style")[["input tokens", "output tokens", "cost $", "seconds"]].mean()
verbosity["output share of cost %"] = (out_df.assign(out_cost=out_df["output tokens"] * PRICE["output"] / 1e6)
                                      .groupby("style").apply(lambda d: d["out_cost"].sum() / d["cost $"].sum() * 100, include_groups=False))
verbosity = verbosity.round({"input tokens": 1, "output tokens": 1, "cost $": 6, "seconds": 2, "output share of cost %": 1})
verbosity

,input tokens,output tokens,cost $,seconds,output share of cost %
style,,,,,
concise instruction,22.0,74.2,0.000393,1.22,94.4
no instruction,15.0,368.8,0.001859,4.20,99.2


## 4. What fits in the context window

Claude Haiku 4.5 has a **200,000-token** context window. How many of these tickets fit, depending on the format?
(One window per request; you'd still pay for every token you send.)

In [5]:
per_ticket = fmt.assign(**{"tokens per ticket": (fmt["tokens"] / len(TICKETS)).round(1)})
per_ticket["tickets per 200K window"] = (CONTEXT / per_ticket["tokens per ticket"]).astype(int)
per_ticket["$ to send one full window"] = round(CONTEXT * PRICE["input"] / 1e6, 2)
per_ticket[["format", "tokens per ticket", "tickets per 200K window", "$ to send one full window"]]

,format,tokens per ticket,tickets per 200K window,$ to send one full window
0,CSV,43.2,4629,0.2
1,Markdown table,52.3,3824,0.2
2,"JSON, minified",69.6,2873,0.2
3,YAML,79.2,2525,0.2
4,"JSON, pretty (indent=2)",104.6,1912,0.2


## 5. Findings

Every number below comes from the cells above; these are the numbers the LinkedIn post uses.

In [6]:
f = {
    "model": MODEL, "sdk": anthropic.__version__, "tickets": len(TICKETS),
    "formats": fmt.to_dict("records"),
    "chat_turns": chat.to_dict("records"), "chat_totals": totals,
    "chat_cost_total": round(float(chat["cost $"].sum()), 6),
    "verbosity": verbosity.reset_index().to_dict("records"),
    "fits": per_ticket[["format", "tokens per ticket", "tickets per 200K window"]].to_dict("records"),
}
Path("findings.json").write_text(json.dumps(f, indent=2, ensure_ascii=False, default=str))
print(json.dumps({k: f[k] for k in ("formats", "chat_totals", "verbosity", "fits")}, indent=1, default=str))

{
 "formats": [
  {
   "format": "CSV",
   "characters": 2749,
   "tokens": 863,
   "vs cheapest": 1.0
  },
  {
   "format": "Markdown table",
   "characters": 3127,
   "tokens": 1046,
   "vs cheapest": 1.21
  },
  {
   "format": "JSON, minified",
   "characters": 4827,
   "tokens": 1393,
   "vs cheapest": 1.61
  },
  {
   "format": "YAML",
   "characters": 4810,
   "tokens": 1584,
   "vs cheapest": 1.84
  },
  {
   "format": "JSON, pretty (indent=2)",
   "characters": 5908,
   "tokens": 2091,
   "vs cheapest": 2.42
  }
 ],
 "chat_totals": {
  "input tokens, full history": 18441,
  "input tokens, last 2 turns only": 7261,
  "turn 1 input": 46,
  "turn 10 input": 3660,
  "saving %": 61
 },
 "verbosity": [
  {
   "style": "concise instruction",
   "input tokens": 22.0,
   "output tokens": 74.2,
   "cost $": 0.000393,
   "seconds": 1.22,
   "output share of cost %": 94.4
  },
  {
   "style": "no instruction",
   "input tokens": 15.0,
   "output tokens": 368.8,
   "cost $": 0.001859,
   "s